In [1]:
import gymnasium as gym

# 1. 创建经典平衡倒立摆游戏 CartPole
# 如果想弹出动画窗口看画面，可以加上 render_mode="human"
env = gym.make("CartPole-v1")

# 2. 开局重置，获取初始状态
state, info = env.reset()
print(f"初始状态 (车位置, 车速度, 杆角度, 杆角速度): \n{state}\n")

total_reward = 0
for step in range(10):
    # 3. 产生一个动作：0 代表往左推，1 代表往右推 (这里先随机选一个动作)
    action = env.action_space.sample()
    
    # 4. 环境推进一步
    next_state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    
    print(f"第 {step+1} 步 -> 动作: {action} (0左/1右), 获得奖励: {reward}")
    
    # 如果杆子倒了或者小车出界，提前结束
    if terminated or truncated:
        print("本局结束！")
        break

print(f"\n本局累计得分: {total_reward}")
env.close()

初始状态 (车位置, 车速度, 杆角度, 杆角速度): 
[-0.02611976  0.00245066 -0.04987887  0.02535782]

第 1 步 -> 动作: 0 (0左/1右), 获得奖励: 1.0
第 2 步 -> 动作: 1 (0左/1右), 获得奖励: 1.0
第 3 步 -> 动作: 0 (0左/1右), 获得奖励: 1.0
第 4 步 -> 动作: 0 (0左/1右), 获得奖励: 1.0
第 5 步 -> 动作: 0 (0左/1右), 获得奖励: 1.0
第 6 步 -> 动作: 1 (0左/1右), 获得奖励: 1.0
第 7 步 -> 动作: 0 (0左/1右), 获得奖励: 1.0
第 8 步 -> 动作: 0 (0左/1右), 获得奖励: 1.0
第 9 步 -> 动作: 1 (0左/1右), 获得奖励: 1.0
第 10 步 -> 动作: 0 (0左/1右), 获得奖励: 1.0

本局累计得分: 10.0


In [7]:
import gymnasium as gym

env = gym.make("CartPole-v1", render_mode="human")
state, info = env.reset()

step = 0
total_reward = 0

# 让它一直玩，直到杆子倒下
while True:
    step += 1
    action = env.action_space.sample() # 依然是随机乱按
    next_state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    
    # 只要倾斜角度过大或者小车跑出屏幕边界，terminated 就会变成 True
    if terminated or truncated:
        print(f"哎呀杆子倒了！纯靠瞎蒙一共坚持了: {step} 步，得分: {total_reward}")
        break

env.close()

哎呀杆子倒了！纯靠瞎蒙一共坚持了: 15 步，得分: 15.0


In [8]:
import gymnasium as gym
import time

# 开启窗口渲染
env = gym.make("CartPole-v1", render_mode="human")

# 让它连续玩 3 局
for episode in range(3):
    state, info = env.reset()
    total_reward = 0
    step = 0
    print(f"\n--- 第 {episode + 1} 局开始 ---")
    
    while True:
        step += 1
        # 故意放慢动作：每走一步暂停 0.05 秒（相当于每秒20帧），方便肉眼观察
        time.sleep(0.05)
        
        # 随机挑选动作
        action = env.action_space.sample()
        next_state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        
        # 倒了就结束本局
        if terminated or truncated:
            print(f"第 {episode + 1} 局结束！坚持了 {step} 步")
            time.sleep(0.5) # 倒下后稍微停顿半秒再开下一局
            break

# 3局全部玩完后再关闭窗口
env.close()


--- 第 1 局开始 ---
第 1 局结束！坚持了 15 步

--- 第 2 局开始 ---
第 2 局结束！坚持了 17 步

--- 第 3 局开始 ---
第 3 局结束！坚持了 18 步


In [10]:
import gymnasium as gym
env = gym.make('MountainCar-v0')
print('观测空间 = {}'. format(env.observation_space))
print('动作空间 = {}'. format(env.action_space))
print('观测范围 = {} ~ {}'. format(env.observation_space.low,
env.observation_space.high))
print('动作数 = {}'. format(env.action_space.n))

观测空间 = Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
动作空间 = Discrete(3)
观测范围 = [-1.2  -0.07] ~ [0.6  0.07]
动作数 = 3


In [12]:
class SimpleAgent:
    def __init__(self, env):
        pass

    def decide(self, observation):  # 决策
        position, velocity = observation
        lb = min(-0.09 * (position + 0.25) ** 2 + 0.03,
                 0.3 * (position + 0.9) ** 4 - 0.008)
        ub = -0.07 * (position + 0.38) ** 2 + 0.07
        if lb < velocity < ub:
            action = 2
        else:
            action = 0
        return action  # 返回动作

    def learn(self, *args):  # 学习
        pass

agent = SimpleAgent(env)

In [13]:
def play(env, agent, render=False, train=False):
    episode_reward = 0.
    res = env.reset()
    # 兼容新老版本的 reset
    observation = res[0] if isinstance(res, tuple) else res
    
    while True:
        if render:
            env.render()
        action = agent.decide(observation)
        step_result = env.step(action)
        
        # 兼容老版 4 个返回值和新版 5 个返回值
        if len(step_result) == 5:
            next_observation, reward, terminated, truncated, _ = step_result
            done = terminated or truncated
        else:
            next_observation, reward, done, _ = step_result
            
        episode_reward += reward
        if train:
            agent.learn(observation, action, reward, done)
        if done:
            break
        observation = next_observation
    return episode_reward

In [20]:
import gymnasium as gym

# 关键就在这里：加上 render_mode="human" 告诉它弹窗显示
env = gym.make("MountainCar-v0", render_mode="human")
agent = SimpleAgent(env)

# 开始玩，这次就会弹窗了！
episode_reward = play(env, agent, render=True)
print('回合奖励 = {}'.format(episode_reward))

# 玩完记得关掉窗口
env.close()

回合奖励 = -104.0


In [19]:
import numpy as np

# 注意：评估100次时不要加 render_mode="human"，不弹窗能以极快速度（不到1秒）跑完！
env = gym.make("MountainCar-v0")
agent = SimpleAgent(env)

# 连续玩 100 局，记录每一局的总得分
episode_rewards = [play(env, agent) for _ in range(100)]

# 计算 100 局的平均分
print('平均回合奖励 = {}'.format(np.mean(episode_rewards)))

env.close()

平均回合奖励 = -104.24
